# Infer independent context and population scores (Production)

Production inference workflow for the V1 representation learning model. Evaluates normal context consistency with EMA latent prediction ($S_{\mathrm{pred}}$) and normal population distance ($S_{\mathrm{pop}}$).

Key features:

- Direct parameters: configure inference directly in the notebook via `InferenceParams` class arguments (e.g. `params = InferenceParams(batch_size=64, mad_multiplier=2.5, in_memory=True)`).
- In-memory caching & streaming: set `in_memory=True` to preload samples into Host RAM for fastest evaluation, or `in_memory=False` for $\mathcal{O}(1)$ disk streaming on $100\text{K}+$ datasets.
- Progress tracking: integrated `tqdm` progress bars for batch scoring across validation and test splits.
- Production checkpoint: restores the trained model weights and normal reference bank from the published Kaggle Model `trietp1253201581/v1-representation-20260904-01` (`pyTorch/default/1`, file `v1_representation_20260904_01.pt`); override with `V1_CHECKPOINT_PATH`, local default `checkpoints/v1_representation.pt`.
- Compute device: automatically uses CUDA when available (`torch.cuda.is_available()`), with transparent CPU fallback.
- Sharded dataset: reads persisted train, validation, and test shards through `FileDataset`.
- Normal-only reference bank: fitted from the persisted normal-only train split; validation and test labels are not used for fitting.
- Independent scoring: computes file-level scores $S_{\mathrm{pred}}$ and $S_{\mathrm{pop}}$, independent MAD thresholds, and timestep-level anomaly localization.
- Evaluation: assesses anomaly detection performance against test split ground truth when anomaly metadata is present.


In [1]:
from copy import deepcopy
from dataclasses import dataclass
from itertools import islice
import json
import os
from pathlib import Path
import sys
import torch
import zipfile
from tqdm.auto import tqdm

# Unpack the bundled src.zip into a Kaggle-writable path, with a local fallback.
_WORKING_SRC = Path('/kaggle/working/src')
_ARCHIVE_CANDIDATES = [Path.cwd() / 'src.zip', Path('src.zip')]
_INPUT_ROOT = Path('/kaggle/input')
if _INPUT_ROOT.is_dir():
    _ARCHIVE_CANDIDATES.extend(sorted(_INPUT_ROOT.glob('*/src.zip')))
    _ARCHIVE_CANDIDATES.extend(sorted(_INPUT_ROOT.glob('*/*/src.zip')))
for _ARCHIVE in _ARCHIVE_CANDIDATES:
    if _ARCHIVE.is_file() and not _WORKING_SRC.is_dir():
        _WORKING_SRC.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(_ARCHIVE) as _zip:
            _zip.extractall(_WORKING_SRC.parent)
        break

# Unified environment detection: check Kaggle dataset paths first, then local paths
candidates = [
    Path('/kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01/src'),
    Path('/kaggle/input/anomaly-representation-20260903-01/src'),
    _WORKING_SRC,
    *Path('/kaggle/input').glob('**/src'),
    Path.cwd() / 'src',
    Path.cwd().parent / 'src',
    Path.cwd().parent.parent / 'src',
]
found_src = False
for candidate in candidates:
    if (candidate / 'representation').is_dir():
        sys.path.insert(0, str(candidate))
        print(f"[Env] Loaded representation modules from: {candidate}")
        found_src = True
        break
if not found_src:
    print("[Env] Warning: Could not locate 'src/representation' in candidate paths.")

# Published V1 checkpoint (Kaggle Model, version-pinned; never bundled).
V1_MODEL_HANDLE = 'trietp1253201581/v1-representation-20260904-01/pyTorch/default/1'
V1_CHECKPOINT_FILENAME = 'v1_representation_20260904_01.pt'
V1_MODEL_MOUNT = Path(f'/kaggle/input/v1-representation-20260904-01/pyTorch/default/1/{V1_CHECKPOINT_FILENAME}')

def _resolve_checkpoint(local_default='checkpoints/v1_representation.pt'):
    """Resolve the user-supplied V1 checkpoint: explicit override, attached Kaggle Model, then local default."""
    override = os.environ.get('V1_CHECKPOINT_PATH') or os.environ.get('V1_CHECKPOINT')
    if override:
        return Path(override).expanduser()
    if V1_MODEL_MOUNT.is_file():
        return V1_MODEL_MOUNT
    if Path('/kaggle/input').is_dir():
        matches = sorted(Path('/kaggle/input').rglob(V1_CHECKPOINT_FILENAME))
        if matches:
            return matches[0]
    return Path(local_default)

# Support Kaggle auto-unzipped directories (where shard-XXXXX.zip is extracted to folder shard-XXXXX/)
import synth.dataset
if Path('/kaggle/input').is_dir() or os.environ.get('V1_SKIP_STRICT_HASH', '0') == '1':
    synth.dataset._verify_shard = lambda root, shard, **kw: True
    
    def _kaggle_auto_unzip_iter(output_dir, split=None):
        root = Path(output_dir)
        manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
        names = [split] if split else list(manifest['splits'])
        for name in names:
            if name not in manifest['splits']:
                raise ValueError(f"split {name!r} is absent from manifest")
            for shard in manifest['splits'][name]['shards']:
                shard_rel = str(shard['path'])
                candidates_dir = [
                    root / (shard_rel[:-4] if shard_rel.endswith('.zip') else shard_rel),
                    root / shard_rel,
                    root / name / Path(shard_rel).stem,
                ]
                shard_dir = next((d for d in candidates_dir if d.is_dir()), None)
                if shard_dir is not None:
                    for file_id in shard['file_ids']:
                        npz_file = shard_dir / f"{file_id}.npz"
                        if npz_file.is_file():
                            yield synth.dataset.load_sample(npz_file)
                        else:
                            matches = list(shard_dir.glob(f"**/{file_id}.npz"))
                            if matches:
                                yield synth.dataset.load_sample(matches[0])
                    continue
                # Standard zip file fallback
                shard_path = root / shard_rel
                if shard_path.is_file():
                    with zipfile.ZipFile(shard_path) as archive:
                        for file_id in shard['file_ids']:
                            yield synth.dataset.load_sample_bytes(archive.read(f"{file_id}.npz"))
                            
    synth.dataset.iter_materialized = _kaggle_auto_unzip_iter
    print("[Env] Kaggle mount detected: unzipped directory support enabled.")

from representation import V1Config
from representation.checkpoint import load_checkpoint, save_checkpoint
from representation.data import FileDataset, collate_variable_files
from representation.inference import NormalReferenceBank, RepresentationInference, mad_threshold
from representation.model import V1RepresentationModel
from representation.trainer import RepresentationTrainer
from synth.config import PatchConfig
from synth.patchify import Patchifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Hardware] Compute device:', device)
if device.type == 'cuda':
    print('[Hardware] CUDA device name:', torch.cuda.get_device_name(0))
    print('[Hardware] Allocated memory:', f"{torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")


[Env] Loaded representation modules from: /kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01/src/src
[Env] Kaggle mount detected: unzipped directory support enabled.
[Hardware] Compute device: cuda
[Hardware] CUDA device name: Tesla T4
[Hardware] Allocated memory: 0.0 MB


In [3]:
def _resolve_checkpoint_path() -> str:
    """Checkpoint resolution shared with the model contract: explicit override, attached model, local default."""
    return str(_resolve_checkpoint())


def _resolve_default_paths() -> tuple[str, str]:
    """Auto-detect Kaggle environment vs local server environment."""
    # 1. Check if running on Kaggle
    kaggle_exact = Path('/kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01')
    if (kaggle_exact / 'manifest.json').is_file() or (kaggle_exact / 'test').is_dir():
        print(f"[Env] Detected Kaggle dataset mount: {kaggle_exact}")
        return str(kaggle_exact), _resolve_checkpoint_path()

    kaggle_alt = Path('/kaggle/input/anomaly-representation-20260903-01')
    if (kaggle_alt / 'manifest.json').is_file():
        print(f"[Env] Detected Kaggle dataset mount: {kaggle_alt}")
        return str(kaggle_alt), _resolve_checkpoint_path()

    if Path('/kaggle/input').is_dir():
        for p in Path('/kaggle/input').glob('**/manifest.json'):
            print(f"[Env] Discovered Kaggle dataset at: {p.parent}")
            return str(p.parent), _resolve_checkpoint_path()

    # 2. Running on Local Server
    local_data = os.environ.get('V1_DATA_ROOT', 'data/generated/production')
    print(f"[Env] Detected Local/Server environment (data_root={local_data})")
    return local_data, _resolve_checkpoint_path()


_default_data, _default_ckpt = _resolve_default_paths()

@dataclass
class InferenceParams:
    """Unified inference configuration (works identically on Local Server and Kaggle GPU).
    
    Modify parameters directly here or pass keyword arguments to InferenceParams(...).
    """
    # Dataset and paths (auto-resolved based on environment)
    data_root: str = _default_data
    checkpoint_path: str = _default_ckpt
    max_samples: int | None = int(os.environ['V1_MAX_SAMPLES']) if 'V1_MAX_SAMPLES' in os.environ else None
    
    # In-memory RAM caching option
    in_memory: bool = os.environ.get('V1_IN_MEMORY', 'true').lower() in ('true', '1', 'yes')
    
    # Inference hyperparameters (optimized defaults for GPU)
    batch_size: int = int(os.environ.get('V1_BATCH_SIZE', '64'))
    mad_multiplier: float = float(os.environ.get('V1_MAD_MULTIPLIER', '2.5'))

# Direct instantiation: edit parameters here directly when running interactively
params = InferenceParams()

configured_root = Path(params.data_root).expanduser()
repo_root = Path.cwd()
while repo_root.name in ('notebooks', 'local', 'kaggle') and not (repo_root / 'src' / 'representation').is_dir():
    repo_root = repo_root.parent
if not (repo_root / 'src' / 'representation').is_dir() and (repo_root.parent / 'src' / 'representation').is_dir():
    repo_root = repo_root.parent
DATA_ROOT = configured_root if configured_root.is_absolute() else repo_root / configured_root
MANIFEST_PATH = DATA_ROOT / 'manifest.json'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"V1 dataset manifest not found at {MANIFEST_PATH}. Run uv run python -m synth.cli --output {DATA_ROOT} first.")
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
required_splits = ('train', 'val', 'test')
splits = manifest.get('splits')
if not isinstance(splits, dict):
    raise RuntimeError(f"V1 dataset manifest at {MANIFEST_PATH} has no split mapping; regenerate with uv run python -m synth.cli.")
missing_splits = [name for name in required_splits if name not in splits]
if missing_splits:
    raise RuntimeError(f"V1 dataset at {DATA_ROOT} is missing required splits: {', '.join(missing_splits)}. Regenerate with uv run python -m synth.cli.")

def load_split(name: str, limit: int = 4):
    entry = splits[name]
    if not isinstance(entry, dict) or entry.get('status') != 'complete':
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is not complete; rerun uv run python -m synth.cli --output {DATA_ROOT} --resume.")
    samples = list(islice(FileDataset(DATA_ROOT, split=name), limit))
    if not samples:
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is empty.")
    return samples

train_samples = load_split('train')
val_samples = load_split('val', limit=2)
test_samples = load_split('test', limit=2)
print('dataset root', DATA_ROOT, 'manifest counts', manifest['counts'])
print('train IDs', [sample.file_id for sample in train_samples], 'val IDs', [sample.file_id for sample in val_samples], 'test IDs', [sample.file_id for sample in test_samples])
print(f"Configured parameters: batch_size={params.batch_size}, mad_multiplier={params.mad_multiplier}, in_memory={params.in_memory}")
print(f'Model {params.checkpoint_path}')

[Env] Detected Kaggle dataset mount: /kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01
dataset root /kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01 manifest counts {'test': 20000, 'train': 25000, 'val': 5000}
train IDs ['N-train-51c5413776', 'N-train-89c49457e8', 'N-train-8daf1efbae', 'N-train-5d151c3e57'] val IDs ['N-val-3a33ca884c', 'N-val-a1b3618996'] test IDs ['N-test-59a7dd93e8', 'N-test-b4ae0abfda']
Configured parameters: batch_size=64, mad_multiplier=2.5, in_memory=True
Model /kaggle/input/models/trietp1253201581/v1-representation-20260904-01/pytorch/default/1/v1_representation_20260904_01.pt


In [4]:
configured_ckpt = Path(params.checkpoint_path).expanduser()
checkpoint_path = configured_ckpt if configured_ckpt.is_absolute() else repo_root / configured_ckpt

if checkpoint_path.is_file():
    payload = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    saved_cfg = payload.get('config', {})
    cfg = V1Config(**saved_cfg)
    print(f"Loaded configuration from {checkpoint_path}: d_model={cfg.d_model}, layers={cfg.sequence_layers}, heads={cfg.attention_heads}")
else:
    cfg = V1Config(
        n_channels=train_samples[0].C,
        patch_size=32,
        stride=16,
        d_model=int(os.environ.get('V1_D_MODEL', '128')),
        sequence_layers=int(os.environ.get('V1_SEQ_LAYERS', '4')),
        attention_heads=int(os.environ.get('V1_ATTN_HEADS', '4')),
        dropout=0.0,
    )
    print(f"Checkpoint not found at {checkpoint_path}; using configured defaults (d_model={cfg.d_model})")

patchifier = Patchifier(PatchConfig(patch_size=cfg.patch_size, stride=cfg.stride, pad_end=True))

class StreamingBatchDataset:
    """Yield collated minibatches, with optional Host RAM caching and full shuffling."""
    def __init__(self, data_root, split, patchifier, config, b_size=64, max_count=None, base_seed=0, in_memory=True):
        self.data_root = data_root
        self.split = split
        self.patchifier = patchifier
        self.config = config
        self.batch_size = b_size
        self.max_count = max_count
        self.base_seed = base_seed
        self.in_memory = in_memory
        self.samples = None
        
        if in_memory:
            dataset = FileDataset(self.data_root, split=self.split)
            iterator = islice(dataset, self.max_count) if self.max_count is not None else iter(dataset)
            total = manifest['counts'][split] if self.max_count is None else min(self.max_count, manifest['counts'][split])
            self.samples = list(tqdm(iterator, total=total, desc=f"Loading {split} to RAM"))

    def __iter__(self):
        if self.samples is not None:
            chunk = []
            batch_idx = 0
            for sample in self.samples:
                chunk.append(sample)
                if len(chunk) == self.batch_size:
                    yield collate_variable_files(
                        chunk,
                        self.patchifier,
                        masking_config=self.config,
                        masking_seed=self.base_seed + batch_idx,
                    )
                    chunk = []
                    batch_idx += 1
            if chunk:
                yield collate_variable_files(
                    chunk,
                    self.patchifier,
                    masking_config=self.config,
                    masking_seed=self.base_seed + batch_idx,
                )
        else:
            dataset = FileDataset(self.data_root, split=self.split)
            iterator = islice(dataset, self.max_count) if self.max_count is not None else iter(dataset)
            chunk = []
            batch_idx = 0
            for sample in iterator:
                chunk.append(sample)
                if len(chunk) == self.batch_size:
                    yield collate_variable_files(
                        chunk,
                        self.patchifier,
                        masking_config=self.config,
                        masking_seed=self.base_seed + batch_idx,
                    )
                    chunk = []
                    batch_idx += 1
            if chunk:
                yield collate_variable_files(
                    chunk,
                    self.patchifier,
                    masking_config=self.config,
                    masking_seed=self.base_seed + batch_idx,
                )

reference_batches = StreamingBatchDataset(DATA_ROOT, 'train', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=5, in_memory=params.in_memory)
val_batches = StreamingBatchDataset(DATA_ROOT, 'val', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=6, in_memory=params.in_memory)
test_batches = StreamingBatchDataset(DATA_ROOT, 'test', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=7, in_memory=params.in_memory)

reference_batch = next(iter(reference_batches))
val_batch = next(iter(val_batches))
test_batch = next(iter(test_batches))
print('reference signals', tuple(reference_batch['signals'].shape), 'validation signals', tuple(val_batch['signals'].shape), 'test signals', tuple(test_batch['signals'].shape))

Loaded configuration from /kaggle/input/models/trietp1253201581/v1-representation-20260904-01/pytorch/default/1/v1_representation_20260904_01.pt: d_model=128, layers=4, heads=4


Loading train to RAM:   0%|          | 0/25000 [00:00<?, ?it/s]

Loading val to RAM:   0%|          | 0/5000 [00:00<?, ?it/s]

Loading test to RAM:   0%|          | 0/20000 [00:00<?, ?it/s]

reference signals (64, 6, 797) validation signals (64, 6, 798) test signals (64, 6, 800)


In [5]:
model = V1RepresentationModel(cfg, patchifier=patchifier)
model.to(device)
model.eval()

bank = NormalReferenceBank(k=min(cfg.knn_k, manifest['counts']['train']))
if checkpoint_path.is_file():
    meta = load_checkpoint(checkpoint_path, model, reference_bank=bank)
    print(f"Restored checkpoint from {checkpoint_path} (step {meta['step']}, reference bank restored: {meta['has_reference_bank']})")

ref_batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in reference_batch.items()}
with torch.no_grad():
    reference_output = model(ref_batch_device)
if bank.embeddings is None:
    bank.fit(reference_output['file_embedding'])
else:
    bank.fit(reference_output['file_embedding'])

inference = RepresentationInference(model, bank, patchifier, masking_config=cfg)

scored_splits = {}
for split, batch_loader in (('val', val_batches), ('test', test_batches)):
    all_pred = []
    all_pop = []
    all_is_anomalous = []
    all_families = []
    sample_timesteps = []
    
    total_files = len(batch_loader.samples) if batch_loader.samples is not None else (
        manifest['counts'][split] if params.max_samples is None else min(params.max_samples, manifest['counts'][split])
    )
    total_b = (total_files + params.batch_size - 1) // params.batch_size
    pbar = tqdm(batch_loader, total=total_b, desc=f"Scoring {split} ({total_files} files)")
    for batch in pbar:
        scores = inference.score_batch(batch)
        all_pred.extend(scores['S_pred'].cpu().tolist())
        all_pop.extend(scores['S_pop'].cpu().tolist())
        if 'file_labels' in batch and batch['file_labels'] is not None:
            all_is_anomalous.extend([getattr(l, 'value', str(l)).lower() == 'abnormal' for l in batch['file_labels']])
        if 'anomaly_meta' in batch and batch['anomaly_meta'] is not None:
            all_families.extend([meta.family.value if meta is not None else 'normal' for meta in batch['anomaly_meta']])
        if len(sample_timesteps) < 5:
            sample_timesteps.extend(scores['timestep_scores'][:5 - len(sample_timesteps)])
            
    pred_t = torch.tensor(all_pred)
    pop_t = torch.tensor(all_pop)
    t_pred = mad_threshold(pred_t, multiplier=params.mad_multiplier)
    t_pop = mad_threshold(pop_t, multiplier=params.mad_multiplier)
    scored_splits[split] = {
        'S_pred': pred_t,
        'S_pop': pop_t,
        'timestep_scores': sample_timesteps,
        'thresh_pred': t_pred,
        'thresh_pop': t_pop,
        'is_anomalous': all_is_anomalous,
        'families': all_families,
    }
    print(split, 'S_pred', scores['S_pred'].tolist(), 'S_pop', scores['S_pop'].tolist(), 'timestep localization', scores['timestep_scores'][0].tolist())
    print(f"[{split}] scored {len(all_pred)} files | S_pred: mean={pred_t.mean():.4f}, threshold={t_pred:.4f} | S_pop: mean={pop_t.mean():.4f}, threshold={t_pop:.4f}")

print('normal train reference rows', bank.embeddings.shape[0], 'labels were not passed to bank.fit')

Restored checkpoint from /kaggle/input/models/trietp1253201581/v1-representation-20260904-01/pytorch/default/1/v1_representation_20260904_01.pt (step 7840, reference bank restored: True)


Scoring val (5000 files):   0%|          | 0/79 [00:00<?, ?it/s]

val S_pred [0.05848533660173416, 0.08844447135925293, 0.09203015267848969, 0.08939643949270248, 0.04618784040212631, 0.03720983862876892, 0.048515766859054565, 0.06818197667598724] S_pop [5.719466686248779, 4.866408348083496, 5.352347373962402, 5.105682849884033, 5.981816291809082, 5.3575873374938965, 6.292577743530273, 5.624336242675781] timestep localization [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0.01892266236245632, 0

Scoring test (20000 files):   0%|          | 0/313 [00:00<?, ?it/s]

test S_pred [0.059850454330444336, 0.057407960295677185, 0.027388354763388634, 0.03438713401556015, 0.09448089450597763, 0.037785228341817856, 0.04882153123617172, 0.031920239329338074, 0.08004786819219589, 0.049147844314575195, 0.07062295824289322, 0.07667989283800125, 0.046763237565755844, 0.0368695966899395, 0.04158761724829674, 0.049784738570451736, 0.06029414013028145, 0.043193310499191284, 0.04629256948828697, 0.029960431158542633, 0.0407785065472126, 0.11688309907913208, 0.03751164302229881, 0.024115927517414093, 0.0396186038851738, 0.039148639887571335, 0.05372141674160957, 0.03696269169449806, 0.04144071787595749, 0.03712523728609085, 0.060776688158512115, 0.03195981681346893] S_pop [7.484811305999756, 5.605027675628662, 5.477485656738281, 5.790417671203613, 6.397148132324219, 5.385376930236816, 5.952176094055176, 5.360167026519775, 5.305556774139404, 6.276958465576172, 5.310395240783691, 5.705502510070801, 5.679149150848389, 5.188011169433594, 6.117544651031494, 6.05885410308

In [6]:
# Production anomaly detection performance evaluation on test set
from sklearn.metrics import roc_auc_score

test_anomalous_gt = scored_splits.get('test', {}).get('is_anomalous', [])
n_abnormal = sum(test_anomalous_gt)
n_normal = len(test_anomalous_gt) - n_abnormal
print(f"Ground truth distribution in test set: {n_normal} normal, {n_abnormal} abnormal (total {len(test_anomalous_gt)})")

if n_abnormal > 0 and n_normal > 0:
    test_s_pred = scored_splits['test']['S_pred']
    test_s_pop = scored_splits['test']['S_pop']
    
    # Validation-calibrated thresholds
    val_s_pred = scored_splits.get('val', {}).get('S_pred', test_s_pred)
    val_s_pop = scored_splits.get('val', {}).get('S_pop', test_s_pop)
    th_pred = mad_threshold(val_s_pred, multiplier=params.mad_multiplier)
    th_pop = mad_threshold(val_s_pop, multiplier=params.mad_multiplier)

    det_pred = (test_s_pred > th_pred).tolist()
    det_pop = (test_s_pop > th_pop).tolist()
    det_any = [bool(p or q) for p, q in zip(det_pred, det_pop)]

    tp = sum(1 for d, gt in zip(det_any, test_anomalous_gt) if d and gt)
    fp = sum(1 for d, gt in zip(det_any, test_anomalous_gt) if d and not gt)
    fn = sum(1 for d, gt in zip(det_any, test_anomalous_gt) if not d and gt)
    tn = sum(1 for d, gt in zip(det_any, test_anomalous_gt) if not d and not gt)
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, tp + fn)
    f1 = 2 * prec * rec / max(1e-6, prec + rec)
    
    auc_pred = float(roc_auc_score(test_anomalous_gt, test_s_pred.tolist()))
    auc_pop = float(roc_auc_score(test_anomalous_gt, test_s_pop.tolist()))
    
    print(f"Decision thresholds (multiplier={params.mad_multiplier}): th_pred={th_pred:.4f}, th_pop={th_pop:.4f}")
    print(f"Test classification: TP={tp}, FP={fp}, FN={fn}, TN={tn}")
    print(f"Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
    print(f"AUROC: S_pred={auc_pred:.4f}, S_pop={auc_pop:.4f}")
    
    # Anomaly family breakdown
    families = scored_splits.get('test', {}).get('families', [])
    if families:
        detected_by_family = {}
        for fam, is_det in zip(families, det_any):
            if fam != 'normal':
                detected_by_family.setdefault(fam, []).append(is_det)
        for fam, det_list in sorted(detected_by_family.items()):
            print(f"Family {fam:28s}: detected {sum(det_list)}/{len(det_list)}")
else:
    print("Test split does not contain both normal and abnormal files; evaluation metrics skipped.")

Ground truth distribution in test set: 10000 normal, 10000 abnormal (total 20000)
Decision thresholds (multiplier=2.5): th_pred=0.0772, th_pop=6.4668
Test classification: TP=2299, FP=1934, FN=7701, TN=8066
Precision: 0.5431, Recall: 0.2299, F1: 0.3231
AUROC: S_pred=0.5384, S_pop=0.5094
Family contextual_replacement      : detected 222/1112
Family cross_channel_inconsistency : detected 221/1111
Family duration_anomaly            : detected 426/1111
Family freq_phase_mismatch         : detected 217/1111
Family missing_event               : detected 264/1111
Family over_regularity             : detected 221/1111
Family realistic_stuck             : detected 232/1111
Family subtle_drift                : detected 253/1111
Family wrong_transition            : detected 243/1111
